In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToSpeech
import torch
import soundfile as sf
from pathlib import Path
import IPython.display as ipd

# Define the path to your audio file
audio_file_path = Path("../data/speechocean762/train/audios/000002.wav")
audio_array, sampling_rate = sf.read(str(audio_file_path))

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_vc")
model = SpeechT5ForSpeechToSpeech.from_pretrained("microsoft/speecht5_vc")

inputs = processor(audio=audio_array, sampling_rate=sampling_rate, return_tensors="pt")


Some weights of SpeechT5ForSpeechToSpeech were not initialized from the model checkpoint at microsoft/speecht5_vc and are newly initialized: ['speecht5.encoder.prenet.pos_sinusoidal_embed.weights']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToSpeech
import torch
import soundfile as sf
from pathlib import Path
import IPython.display as ipd
import numpy as np
import librosa

# Define the path to your audio file
audio_file_path = Path("../data/speechocean762/train/audios/000002.wav")
audio_array, sampling_rate = sf.read(str(audio_file_path))

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_vc")
model = SpeechT5ForSpeechToSpeech.from_pretrained("microsoft/speecht5_vc")

inputs = processor(audio=audio_array, sampling_rate=sampling_rate, return_tensors="pt")

speaker_embeddings = torch.randn(1, 512)  # Speaker embedding for voice conversion

# Process the audio
with torch.no_grad():
    input_values = inputs.input_values
    attention_mask = inputs.attention_mask
    
    # Get encoder outputs
    encoder_outputs = model.speecht5.encoder(
        input_values=input_values,
        attention_mask=attention_mask,
        return_dict=True
    )
    
    encoder_hidden_states = encoder_outputs.last_hidden_state
    
    # Create initial output sequence (zeros)
    bsz = input_values.size(0)
    output_sequence = encoder_hidden_states.new_zeros(bsz, 1, model.config.num_mel_bins)
    
    # Set parameters
    minlenratio = 0.0
    maxlenratio = 10.0
    threshold = 0.5
    
    # Calculate min and max lengths
    maxlen = int(encoder_hidden_states.size(1) * maxlenratio / model.config.reduction_factor)
    minlen = int(encoder_hidden_states.size(1) * minlenratio / model.config.reduction_factor)
    
    # FIX: Create proper encoder attention mask with correct dimensions
    # The attention mask should match the encoder hidden states dimensions
    if attention_mask is not None:
        # Get the sequence length from encoder hidden states
        encoder_seq_length = encoder_hidden_states.size(1)
        
        # Resize the attention mask to match expected dimensions
        # First convert to 1D sequence mask if it's not already
        if len(attention_mask.shape) > 2:
            encoder_attention_mask = attention_mask.squeeze()
        else:
            encoder_attention_mask = attention_mask
            
        # Ensure it has the right shape for cross-attention
        # Convert from [batch_size, seq_len] to [batch_size, 1, 1, seq_len]
        encoder_attention_mask = encoder_attention_mask.unsqueeze(1).unsqueeze(2)
        
        # Convert from 0/1 to -inf/0 attention mask
        encoder_attention_mask = (1.0 - encoder_attention_mask) * -10000.0
    else:
        # Create a simple mask that attends to all encoder outputs
        encoder_attention_mask = torch.zeros(
            bsz, 1, 1, encoder_hidden_states.size(1),
            device=encoder_hidden_states.device
        )
    
    # Start generation loop
    spectrogram = []
    past_key_values = None
    idx = 0
    
    while True:
        idx += 1
        
        # Run decoder prenet
        decoder_hidden_states = model.speecht5.decoder.prenet(output_sequence, speaker_embeddings)
        
        # Run decoder
        decoder_out = model.speecht5.decoder.wrapped_decoder(
            hidden_states=decoder_hidden_states[:, -1:],
            attention_mask=None,  # No self-attention mask needed for a single token
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=False,
            return_dict=True,
        )
        
        last_decoder_output = decoder_out.last_hidden_state.squeeze(1)
        past_key_values = decoder_out.past_key_values
        
        # Predict spectrum
        spectrum = model.speech_decoder_postnet.feat_out(last_decoder_output)
        spectrum = spectrum.view(bsz, model.config.reduction_factor, model.config.num_mel_bins)
        spectrogram.append(spectrum)
        
        # Extend output sequence
        new_spectrogram = spectrum[:, -1, :].view(bsz, 1, model.config.num_mel_bins)
        output_sequence = torch.cat((output_sequence, new_spectrogram), dim=1)
        
        # Check stop token probability
        prob = torch.sigmoid(model.speech_decoder_postnet.prob_out(last_decoder_output))
        
        # Check stopping conditions
        if idx < minlen:
            continue
        elif idx >= maxlen:
            break
        elif torch.sum(prob, dim=-1) >= threshold:
            break
    
    # Process generated spectrogram
    spectrograms = torch.cat(spectrogram, dim=1)  # Changed from stack to cat for correct dimension
    final_spectrogram = model.speech_decoder_postnet.postnet(spectrograms)
    
    # Convert from mel spectrogram back to waveform using Griffin-Lim algorithm
    mel_spec_np = final_spectrogram.squeeze().detach().numpy()
    
    # Original audio parameters for matching
    n_fft = 1024
    hop_length = 256
    win_length = 1024
    
    # Reconstruct the waveform from the mel spectrogram using Griffin-Lim
    linear_spec = librosa.feature.inverse.mel_to_stft(
        mel_spec_np, 
        sr=sampling_rate,
        n_fft=n_fft,
        power=1.0
    )
    
    # Use Griffin-Lim to reconstruct the audio signal
    waveform_np = librosa.griffinlim(
        linear_spec,
        hop_length=hop_length,
        win_length=win_length,
        n_iter=32  # More iterations give better quality but take longer
    )
    
    # Normalize the waveform to match input amplitude
    max_amp_orig = np.max(np.abs(audio_array))
    max_amp_new = np.max(np.abs(waveform_np))
    if max_amp_new > 0:
        waveform_np = waveform_np * (max_amp_orig / max_amp_new)
    
    # Save the generated audio
    output_audio_path = Path("../data/output_audio.wav")
    sf.write(str(output_audio_path), waveform_np, samplerate=sampling_rate)
    
    print(f"Audio saved to {output_audio_path}")
    print(f"Waveform shape: {waveform_np.shape}")
    
    # Display the waveform in the notebook
    print("Playing generated audio:")
    display(ipd.Audio(waveform_np, rate=sampling_rate))
    
    # Display the original audio for comparison
    print("Playing original audio:")
    display(ipd.Audio(audio_array, rate=sampling_rate))
    
    # Visualize the waveforms
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(3, 1, 1)
    plt.title("Original Waveform")
    plt.plot(audio_array)
    
    plt.subplot(3, 1, 2)
    plt.title("Generated Waveform")
    plt.plot(waveform_np)
    
    # Also visualize the spectrogram
    plt.subplot(3, 1, 3)
    plt.title("Generated Mel Spectrogram")
    librosa.display.specshow(
        mel_spec_np, 
        sr=sampling_rate, 
        hop_length=hop_length, 
        x_axis='time', 
        y_axis='mel'
    )
    plt.colorbar(format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

Some weights of SpeechT5ForSpeechToSpeech were not initialized from the model checkpoint at microsoft/speecht5_vc and are newly initialized: ['speecht5.encoder.prenet.pos_sinusoidal_embed.weights']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ValueError: too many values to unpack (expected 2)

In [ ]:
# Process input audio

speaker_embeddings = torch.zeros((1, 512))
with torch.no_grad():
    encoder = model.get_encoder()
    encoder_outputs = encoder(inputs["input_values"], return_dict=True)

    spectrogram = model.speecht5.decoder.prenet(
        input_values=torch.zeros((1, 1, model.config.num_mel_bins), device=inputs["input_values"].device),
        speaker_embeddings=speaker_embeddings
    )

ipd.Audio(waveform.squeeze().cpu().numpy(), rate=vocoder.config.sampling_rate)


tensor([[[0.0000e+00, 4.5054e-02, 0.0000e+00, 0.0000e+00, 5.0852e-02,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 8.3834e-03,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 9.2583e-03, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 7.9483e-02, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 6.8907e-02, 9.2389e-03,
          3.6457e-02, 0.0000e+00, 4.1435e-02, 0.0000e+00, 0.0000e+00,
          9.2584e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 1.0030e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 4.3749e-02,
          0.0000e+00, 1.7598e-02, 0.0000e+00, 0.0000e+00, 2.2139e-04,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 4.6270e-02,
          0.0000e+00, 4.1945e-03, 6.6518e-02, 5.4174e-02, 0.0000e+00,
          0.0000e+00

In [18]:
import torch

torch.nn.init.normal_(model.speecht5.encoder.prenet.pos_sinusoidal_embed.weights, mean=0.0, std=0.02)

Parameter containing:
tensor([[ 0.0048,  0.0191,  0.0150,  ...,  0.0162, -0.0055,  0.0313],
        [ 0.0108,  0.0101, -0.0289,  ...,  0.0330,  0.0042, -0.0015],
        [ 0.0192, -0.0007,  0.0140,  ..., -0.0435, -0.0118, -0.0241],
        ...,
        [ 0.0156,  0.0086, -0.0080,  ...,  0.0130, -0.0169,  0.0108],
        [ 0.0114,  0.0005, -0.0073,  ...,  0.0081,  0.0053, -0.0139],
        [-0.0084, -0.0231, -0.0080,  ...,  0.0404,  0.0096, -0.0042]])

In [19]:
model.speecht5.encoder.prenet.pos_sinusoidal_embed.weights

Parameter containing:
tensor([[ 0.0048,  0.0191,  0.0150,  ...,  0.0162, -0.0055,  0.0313],
        [ 0.0108,  0.0101, -0.0289,  ...,  0.0330,  0.0042, -0.0015],
        [ 0.0192, -0.0007,  0.0140,  ..., -0.0435, -0.0118, -0.0241],
        ...,
        [ 0.0156,  0.0086, -0.0080,  ...,  0.0130, -0.0169,  0.0108],
        [ 0.0114,  0.0005, -0.0073,  ...,  0.0081,  0.0053, -0.0139],
        [-0.0084, -0.0231, -0.0080,  ...,  0.0404,  0.0096, -0.0042]])